# 面试问题：RLHF 的 Reward Model 怎样训练，Bradley–Terry Loss 和 KL 约束是什么？

**一句话回答**：对同一 prompt 的 chosen/rejected 响应分别输出标量 reward，用 `-log σ(r_chosen-r_rejected)` 拟合相对偏好；按 prompt/标注者切分，处理 tie、位置偏差和长度捷径，并在人类 holdout 上校准。策略优化不是无限追高 reward，而是在参考策略附近最大化 reward，KL 系数控制收益与分布偏移。

本 Notebook 用 PyTorch 手写偏好损失、RewardModel、训练与校准，并推导离散动作下 KL 正则最优策略。受控特征只用于验证公式与状态，不能把合成集上的高准确率外推成人类偏好质量；真实上线还要覆盖拒答、安全、事实性、不同文化语言和标注者分歧。

In [ ]:
from dataclasses import dataclass
import hashlib, json, math
import numpy as np
import torch
from torch import nn

SEED113=11301; torch.manual_seed(SEED113); rng113=np.random.default_rng(SEED113)
n113,d113=300,6; chosen_x113=torch.randn(n113,d113); rejected_x113=torch.randn(n113,d113); true_w113=torch.tensor([1.4,-1.,.7,.3,-.5,1.1]); raw_margin113=(chosen_x113-rejected_x113)@true_w113; swap113=raw_margin113<0; chosen_x113[swap113],rejected_x113[swap113]=rejected_x113[swap113].clone(),chosen_x113[swap113].clone()
assert chosen_x113.shape==rejected_x113.shape==(300,6)
assert torch.mean(((chosen_x113-rejected_x113)@true_w113>0).float())==1
assert SEED113==11301

## 1. 偏好数据必须绑定同一 prompt

pair 包含 prompt/group ID、两响应特征、偏好/tie、标注者和展示顺序。不能把不同 prompt 的绝对 reward 比较当成偏好；train/test 按 prompt family 切分，避免同问题改写泄漏。采集时随机交换 A/B 位置并记录 propensity。

In [ ]:
@dataclass(frozen=True)
class Pair113:
    prompt_id:str; chosen:int; rejected:int; annotator:str; weight:float=1.
    def __post_init__(self):
        if not self.prompt_id or self.chosen==self.rejected or self.weight<=0: raise ValueError("pair_contract")
pair113=Pair113("p1",0,1,"h1")
assert pair113.chosen==0 and pair113.rejected==1
assert pair113.weight==1.
try: Pair113("",1,1,"h",0); raise AssertionError("bad pair accepted")
except ValueError as e: assert str(e)=="pair_contract"

## 2. Bradley–Terry 只依赖 reward 差

`P(chosen≻rejected)=σ(r_c-r_r)`，NLL 为 `softplus(-(r_c-r_r))`。使用 `logaddexp` 避免大负/正 margin 溢出。差为 0 时 loss 是 `log 2`；margin 越大，loss 越小，但这不代表 reward 的绝对尺度有意义。

In [ ]:
def bt_loss113(rc,rr,weight=None):
    per=torch.logaddexp(torch.zeros_like(rc),-(rc-rr))
    if weight is None: return per.mean()
    return (per*weight).sum()/weight.sum()
assert torch.allclose(bt_loss113(torch.tensor([0.]),torch.tensor([0.])),torch.tensor(math.log(2)),atol=1e-7)
assert bt_loss113(torch.tensor([5.]),torch.tensor([-5.]))<bt_loss113(torch.tensor([1.]),torch.tensor([0.]))
assert torch.isfinite(bt_loss113(torch.tensor([1000.]),torch.tensor([-1000.])))

## 3. Offset 不可辨识，尺度影响置信度

给 chosen/rejected reward 同加常数不改变 loss，因此最后一层 bias 对纯 pair loss 没有梯度约束。把差整体放大能降低可分训练集 loss，却可能让概率过度自信；需要 weight decay、held-out calibration 或 reward normalization，而非解释绝对 reward。

In [ ]:
rc113=torch.tensor([1.,2.]); rr113=torch.tensor([0.,1.]); base_loss113=bt_loss113(rc113,rr113); shifted_loss113=bt_loss113(rc113+99,rr113+99); scaled_loss113=bt_loss113(rc113*3,rr113*3)
assert torch.equal(base_loss113,shifted_loss113)
assert scaled_loss113<base_loss113
assert torch.allclose(torch.sigmoid(rc113-rr113),torch.sigmoid((rc113+7)-(rr113+7)))

## 4. Tie、噪声与标注者一致性不能静默丢弃

tie 可用目标概率 0.5 的 binary cross entropy、单独三分类或降低权重；选择取决于标注协议。重复标注估计一致性，低一致样本送仲裁。展示位置、长度与风格是潜在捷径，应做 counterfactual swap。

In [ ]:
def soft_preference_loss113(margin,target,weight=None):
    per=torch.logaddexp(torch.zeros_like(margin),margin)-target*margin
    return per.mean() if weight is None else (per*weight).sum()/weight.sum()
margins113=torch.tensor([2.,-2.,0.]); targets113=torch.tensor([1.,0.,.5]); tie_loss113=soft_preference_loss113(margins113,targets113)
assert tie_loss113>0 and torch.isfinite(tie_loss113)
assert torch.allclose(soft_preference_loss113(torch.tensor([0.]),torch.tensor([.5])),torch.tensor(math.log(2)),atol=1e-7)
assert soft_preference_loss113(torch.tensor([5.]),torch.tensor([1.]))<soft_preference_loss113(torch.tensor([1.]),torch.tensor([1.]))

## 5. 按 prompt group 切分，平衡长度与展示位置

同一 prompt 的多个 response pair 高度相关，必须整体进入同一 split。训练集中若 chosen 总更长，模型会学长度捷径；报告分长度、领域、风险与 annotator slice，并加入内容相同只改格式的反事实对。

In [ ]:
prompt_ids113=np.array([f"p{i//3}" for i in range(n113)]); groups113=sorted(set(prompt_ids113)); rg113=np.random.default_rng(7); rg113.shuffle(groups113); train_groups113=set(groups113[:80]); test_groups113=set(groups113[80:]); train_idx113=np.array([p in train_groups113 for p in prompt_ids113]); test_idx113=~train_idx113
assert set(prompt_ids113[train_idx113]).isdisjoint(set(prompt_ids113[test_idx113]))
assert train_idx113.sum()==240 and test_idx113.sum()==60
assert train_idx113.dtype==bool and np.all(train_idx113^test_idx113)

## 6. 手写 RewardModel 与训练循环

实际模型通常在最后 token/特殊 reward token 上接 scalar head；padding mask 决定池化位置。这里用线性 `nn.Module` 处理已提取特征，手动更新参数。只看 train pair accuracy 会夸大效果，必须在 prompt-disjoint holdout 验收。

In [ ]:
class RewardModel113(nn.Module):
    def __init__(self,d): super().__init__(); self.weight=nn.Parameter(torch.zeros(d)); self.bias=nn.Parameter(torch.zeros(()))
    def forward(self,x): return x@self.weight+self.bias
rm113=RewardModel113(d113); losses113=[]
train_t113=torch.tensor(train_idx113); test_t113=torch.tensor(test_idx113)
for _ in range(120):
    loss=bt_loss113(rm113(chosen_x113[train_t113]),rm113(rejected_x113[train_t113])); losses113.append(float(loss.detach())); loss.backward()
    with torch.no_grad():
        for p in rm113.parameters(): p-=.25*p.grad; p.grad=None
test_margin113=rm113(chosen_x113[test_t113])-rm113(rejected_x113[test_t113]); test_acc113=float((test_margin113>0).float().mean())
assert losses113[-1]<losses113[0]*.35
assert test_acc113>.9
assert abs(float(rm113.bias))<1e-7

## 7. 校准、reward hacking 与分布外审计

`σ(margin)` 可解释为 pair preference 概率，但需 reliability/Brier 校准。策略会寻找 RM 漏洞，例如堆砌长度、模板或讨好语句；构造保持语义不变的长度/风格 probe、对抗候选和人工抽样。RM 高分不是用户价值真值。

In [ ]:
probs113=torch.sigmoid(test_margin113).detach().numpy(); outcomes113=np.ones_like(probs113); brier113=float(np.mean((probs113-outcomes113)**2)); bins113=np.linspace(.5,1,6); ece113=0.
for lo,hi in zip(bins113[:-1],bins113[1:]):
    m=(probs113>=lo)&(probs113<(hi if hi<1 else hi+1e-9))
    if m.any(): ece113+=m.mean()*abs(probs113[m].mean()-outcomes113[m].mean())
assert 0<=brier113<=1 and 0<=ece113<=1
assert np.all((probs113>=0)&(probs113<=1))
margin_np113=test_margin113.detach().numpy(); order113=np.argsort(margin_np113)
assert np.all(np.diff(probs113[order113])>=0)

## 8. KL 正则把策略限制在参考模型附近

单状态离散动作下最大化 `Eπ[r]-β KL(π||π_ref)` 的闭式最优解是 `π*(a)∝π_ref(a)exp(r(a)/β)`。β 小更追 reward、也更偏离参考；β 大更保守。真实 PPO/RLHF 还需在线估 KL、优势和安全门禁。

In [ ]:
def kl_policy113(pi_ref,reward,beta):
    if beta<=0: raise ValueError("beta_contract")
    logit=np.log(np.asarray(pi_ref,float))+np.asarray(reward,float)/beta; logit-=logit.max(); p=np.exp(logit); return p/p.sum()
pref113=np.array([.6,.3,.1]); reward113=np.array([0.,.5,1.]); tight113=kl_policy113(pref113,reward113,.2); conservative113=kl_policy113(pref113,reward113,5.)
manifest113={"schema":1,"loss":"bradley_terry","split":"prompt_group","position":"randomized","tie":"soft_target","policy_constraint":"forward_kl_to_reference"}; digest113=hashlib.sha256(json.dumps(manifest113,sort_keys=True).encode()).hexdigest()
assert tight113[2]>conservative113[2]
assert np.allclose(tight113.sum(),1) and np.linalg.norm(conservative113-pref113)<np.linalg.norm(tight113-pref113)
assert len(digest113)==64 and manifest113["split"]=="prompt_group"

## 面试总结

回答链路是：**同 prompt 偏好合同 → `softplus(-(r_c-r_r))` → offset/尺度不可辨识 → tie/噪声/位置 → group split → 标量 RewardModel → 人工校准与 hacking probes → reward + reference KL 策略目标**。Reward Model 是可被策略利用的近似代理，必须持续人工审计。

延伸阅读：[InstructGPT](https://arxiv.org/abs/2203.02155)、[PPO](https://arxiv.org/abs/1707.06347)、[Learning to Summarize from Human Feedback](https://arxiv.org/abs/2009.01325)。